# SFT C: medical SFT on the plain base (no DAPT)

Run the cells top to bottom in a **fresh runtime** (Unsloth only; do not run vLLM or TRL here).

| Cell | What it does |
|---|---|
| 1 | Config and flags (base model, DAPT toggle, paths, data-hygiene knobs, training knobs) |
| 2 | Builds the SFT C dataset: your SFT-B source mix, deduplicated, benchmark-contamination-filtered, saved to Drive |
| 3 | Loads `Qwen/Qwen3.5-4B-Base` (DAPT adapter merged only if `USE_DAPT=True`) and attaches the LoRA |
| 4 | Trains (same recipe as SFT A: r=128 / alpha=256, lr 2e-5, effective batch 64, response-only loss) |
| 5 | Fixes adapter key names so the adapter loads with plain PEFT / vLLM (no upload) |

Changes versus the SFT-B data builder (all switchable in cell 1):
* `MAX_COPIES=1`: `all_exhausted` recycles small sources (ARC, GSM8K, MedQA train were repeated many times). Each unique conversation is kept once, and the held-out split is therefore clean.
* `CONTAM_FILTER=True`: drops training rows that share a 13-gram or an exact question line with MedQA-test, MedMCQA-val, PubMedQA-labeled, MMLU-medical, GPQA-Diamond or GSM8K-test.
* GSM8K calculator annotations (`<<...>>`) are stripped; very short MedMCQA explanations are not put inside `<think>`.
* PubMedQA label counts are printed; `PUBMED_YES_NO_RATIO_CAP` can limit the yes-bias of `pqa_artificial`.

Because the mix shifts when duplicates are removed, check the printed composition table before training.


In [ ]:
# Run ONCE in a fresh runtime, then Runtime -> Restart session (keeps /content), then run the cells below.
# flash-linear-attention = fast kernels for Qwen3.5's 24 linear-attention layers. Without it transformers silently falls back
# to a pure-PyTorch path whose backward pass is extremely slow (training sits at step 0).
!pip install -U cut-cross-entropy flash-linear-attention


In [ ]:
import os
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"   # unsloth's torch.compile of the training graph stalled at step 0 for 10+ min; eager runs at full speed
import unsloth  # must be imported before transformers / trl
import re, gc, json, math, time, hashlib, collections
import torch

# ---------------------------------------------------------------- model
BASE_MODEL_ID = "Qwen/Qwen3.5-4B-Base"
USE_DAPT      = False                                              # False = SFT C on the plain base (the point of this notebook)
DAPT_ADAPTER  = "adi274903/Med-Qwen-3.5-4B-Base-DAPT-Adapter"      # only used when USE_DAPT=True (for the matched DAPT arm)

# ---------------------------------------------------------------- paths (new folders: nothing existing is overwritten)
RUN_NAME   = "sft_c_base_noDAPT" if not USE_DAPT else "sft_c_dapt"
DRIVE_ROOT = "/content/drive/MyDrive/Med_Qwen_4B_SFT_C_noDAPT" if not USE_DAPT else "/content/drive/MyDrive/Med_Qwen_4B_SFT_C_DAPT"
CACHE_DIR  = "/content/drive/MyDrive/sft_c_cache"                  # lives on Drive so both arms train on the identical rows
REBUILD_CACHE = False                                              # True = ignore an existing cache and rebuild

# ---------------------------------------------------------------- data
TARGET_SEQ_LEN = 2048
HOLDOUT = 2500
SEED = 3407
MAX_COPIES = 1                  # 1 = every unique conversation once (no recycling). Raise to allow repeats of small sources.
CONTAM_FILTER = True            # drop rows overlapping benchmark test questions
PUBMED_YES_NO_RATIO_CAP = 2.0   # keeps "yes" <= 2x "no" in the PubMedQA-artificial part (it is ~93% yes otherwise); None = off

# ---------------------------------------------------------------- training (SFT A recipe)
MAX_SEQ_LENGTH = 2048
TOTAL_STEPS = 2700
BATCH_SIZE = 4
GRAD_ACCUM = 16                 # effective batch = 64 conversations
LEARNING_RATE = 2e-5
EVAL_STEPS = 300                # SFT A used 150; evaluation costs ~70 s each time
GRAD_CKPT = False                # smoke test ran fine without checkpointing; if you hit CUDA OOM set True

if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
print("run:", RUN_NAME, "| output:", DRIVE_ROOT, "| cache:", CACHE_DIR)


In [ ]:
from datasets import load_dataset, Dataset, load_from_disk
from transformers import AutoTokenizer
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

CACHE_TRAIN_EVAL = os.path.join(CACHE_DIR, "split")
if os.path.isdir(CACHE_TRAIN_EVAL) and not REBUILD_CACHE:
    print("Cache exists, skipping the build:", CACHE_TRAIN_EVAL)
    print(open(os.path.join(CACHE_DIR, "stats.json")).read() if os.path.exists(os.path.join(CACHE_DIR, "stats.json")) else "")
else:
    t_start = time.perf_counter()
    if MAX_COPIES != 1:
        print(f"[warn] this fast builder makes a single pass over each source, so MAX_COPIES={MAX_COPIES} has no effect (behaves as 1).")
    data_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

    # ============================================================ contamination index
    NG = 30                      # long window: 13-grams matched boilerplate (vitals lines, "which of the following..." stems)
    words = lambda s: re.findall(r"[a-z0-9]+", s.lower())
    EVAL_NGRAMS, EVAL_LINES = set(), set()

    def index_eval_text(s):
        w = words(s)
        if len(w) >= NG:
            for i in range(len(w) - NG + 1):
                EVAL_NGRAMS.add(hash(tuple(w[i:i + NG])))
        if len(w) >= 6:
            EVAL_LINES.add(" ".join(w))

    def build_eval_index():
        loaded = []

        def safe(name, fn):
            try:
                loaded.append(f"{name} ({fn()})")
            except Exception as e:  # noqa: BLE001
                print(f"[contam] {name} skipped: {type(e).__name__}: {str(e)[:80]}")

        def medqa():
            ds = load_dataset("GBaker/MedQA-USMLE-4-options", split="test")
            for it in ds: index_eval_text(it["question"])
            return len(ds)

        def medmcqa():
            ds = load_dataset("openlifescienceai/medmcqa", split="validation")
            for it in ds: index_eval_text(it["question"])
            return len(ds)

        def pubmedqa():
            ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
            for it in ds:
                index_eval_text(it["question"])
                index_eval_text(" ".join(it["context"]["contexts"]))
            return len(ds)

        def mmlu():
            n = 0
            for subj in ["clinical_knowledge", "college_medicine", "medical_genetics", "professional_medicine"]:
                ds = load_dataset("cais/mmlu", subj, split="test")
                for it in ds: index_eval_text(it["question"])
                n += len(ds)
            return n

        def gpqa():
            ds = load_dataset("Idavidrein/gpqa", "gpqa_diamond", split="train")
            for it in ds: index_eval_text(it["Question"])
            return len(ds)

        def gsm8k():
            ds = load_dataset("openai/gsm8k", "main", split="test")
            for it in ds: index_eval_text(it["question"])
            return len(ds)

        for name, fn in [("MedQA-test", medqa), ("MedMCQA-val", medmcqa), ("PubMedQA-labeled", pubmedqa),
                         ("MMLU-med", mmlu), ("GPQA-Diamond", gpqa), ("GSM8K-test", gsm8k)]:
            safe(name, fn)
        print(f"[contam] indexed: {', '.join(loaded)} | {len(EVAL_NGRAMS):,} {NG}-grams, {len(EVAL_LINES):,} question lines")

    def contaminated(text):
        for line in text.split("\n"):
            l = line.replace("<|im_end|>", "").strip()      # the chat template glues this onto the last line of a message
            if l.lower().startswith("question:"):
                l = l[9:]
            w = words(l)
            if len(w) >= 6 and " ".join(w) in EVAL_LINES:
                return True
        w = words(text)
        for i in range(len(w) - NG + 1):
            if hash(tuple(w[i:i + NG])) in EVAL_NGRAMS:
                return True
        return False

    if CONTAM_FILTER:
        build_eval_index()

    # ============================================================ sources (SFT-B mix)
    print("Connecting to SFT streams...")
    med_o1_stream = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "en", split="train", streaming=True).take(45_000)
    ii_med_stream = load_dataset("Intelligent-Internet/II-Medical-Reasoning-SFT", split="train", streaming=True).take(50_000)
    medqa_stream = load_dataset("GBaker/MedQA-USMLE-4-options", split="train", streaming=True).take(20_000)
    alpacare_stream = load_dataset("lavita/AlpaCare-MedInstruct-52k", split="train", streaming=True)
    medmcqa_stream = load_dataset("openlifescienceai/medmcqa", split="train", streaming=True).take(50_000)
    pubmedqa_stream = load_dataset("qiaojin/PubMedQA", "pqa_artificial", split="train", streaming=True).take(30_000)
    pubmed_causal_stream = load_dataset("medalpaca/medical_meadow_pubmed_causal", split="train", streaming=True).take(25_000)
    gsm8k_stream = load_dataset("openai/gsm8k", "main", split="train", streaming=True)
    orca_math_stream = load_dataset("microsoft/orca-math-word-problems-200k", split="train", streaming=True).take(25_000)
    arc_stream = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="train", streaming=True)
    ultrachat_stream = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft", streaming=True).take(20_000)

    def clean_string(val):
        return val.strip() if isinstance(val, str) else ""

    def format_med_o1(ex):
        q, cot, res = clean_string(ex.get("Question")), clean_string(ex.get("Complex_CoT")), clean_string(ex.get("Response"))
        content = f"<think>\n{cot}\n</think>\n\n{res}" if cot else res
        return {"messages": [{"role": "user", "content": q}, {"role": "assistant", "content": content}]}

    def format_ii_med(ex):
        conv = ex.get("messages") or ex.get("conversations")
        if conv and isinstance(conv, list):
            messages = []
            for turn in conv:
                role = turn.get("role") or turn.get("from")
                role = "assistant" if role in ("gpt", "assistant") else "user"
                messages.append({"role": role, "content": clean_string(turn.get("content") or turn.get("value"))})
            return {"messages": messages}
        prompt = clean_string(ex.get("instruction") or ex.get("question") or ex.get("input"))
        response = clean_string(ex.get("output") or ex.get("response"))
        return {"messages": [{"role": "user", "content": prompt}, {"role": "assistant", "content": response}]}

    def format_medqa(ex):
        q = clean_string(ex.get("question"))
        opts = ex.get("options", {})
        ans = clean_string(ex.get("answer_idx")).upper()
        if not q or ans not in ("A", "B", "C", "D"):
            return {"messages": []}
        prompt = (f"Question: {q}\nA. {clean_string(opts.get('A'))}\nB. {clean_string(opts.get('B'))}\n"
                  f"C. {clean_string(opts.get('C'))}\nD. {clean_string(opts.get('D'))}\n"
                  "Provide your diagnostic reasoning and the final answer.")
        return {"messages": [{"role": "user", "content": prompt}, {"role": "assistant", "content": f"The correct answer is {ans}."}]}

    def format_alpacare(ex):
        inst, inp, out = clean_string(ex.get("instruction")), clean_string(ex.get("input")), clean_string(ex.get("output"))
        prompt = f"{inst}\n\nCase Details: {inp}" if inp else inst
        return {"messages": [{"role": "user", "content": prompt}, {"role": "assistant", "content": out}]}

    def format_medmcqa(ex):
        ans_letter = {0: "A", 1: "B", 2: "C", 3: "D", "0": "A", "1": "B", "2": "C", "3": "D"}.get(ex.get("cop"))
        if not ans_letter:
            return {"messages": []}
        q = clean_string(ex.get("question"))
        exp = clean_string(ex.get("exp"))
        if len(exp) < 30:                       # empty / one-liner explanations are not reasoning
            exp = ""
        prompt = (f"Question: {q}\nA. {clean_string(ex.get('opa'))}\nB. {clean_string(ex.get('opb'))}\n"
                  f"C. {clean_string(ex.get('opc'))}\nD. {clean_string(ex.get('opd'))}\n"
                  "Provide your clinical reasoning and state the correct option letter.")
        content = f"<think>\n{exp}\n</think>\n\nThe correct answer is {ans_letter}." if exp else f"The correct answer is {ans_letter}."
        return {"messages": [{"role": "user", "content": prompt}, {"role": "assistant", "content": content}]}

    def format_pubmedqa(ex):
        ctx_data = ex.get("context", {})
        contexts = ctx_data.get("contexts", []) if isinstance(ctx_data, dict) else []
        ctx_str = " ".join(contexts) if isinstance(contexts, list) else str(contexts)
        q = clean_string(ex.get("question"))
        decision = clean_string(ex.get("final_decision")).lower()
        long_answer = clean_string(ex.get("long_answer"))
        if decision not in ("yes", "no", "maybe"):
            return {"messages": []}
        prompt = (f"Context: {ctx_str}\n\nQuestion: {q}\n\n"
                  "Based on the provided biomedical abstract, evaluate the findings and answer with 'yes', 'no', or 'maybe'.")
        content = (f"<think>\n{long_answer}\n</think>\n\nBased on the study, the answer is {decision}."
                   if long_answer else f"Based on the study, the answer is {decision}.")
        return {"messages": [{"role": "user", "content": prompt}, {"role": "assistant", "content": content}],
                "src": f"pubmedqa_{decision}"}

    def format_pubmed_causal(ex):
        inst, inp, out = clean_string(ex.get("instruction")), clean_string(ex.get("input")), clean_string(ex.get("output"))
        prompt = f"Biomedical Study Context:\n{inp}\n\nTask: {inst}" if inp else f"Biomedical Inquiry:\n{inst}"
        return {"messages": [{"role": "user", "content": prompt}, {"role": "assistant", "content": out}]}

    def format_gsm8k(ex):
        q, ans = clean_string(ex.get("question")), clean_string(ex.get("answer"))
        if "####" in ans:
            reasoning, final_num = ans.rsplit("####", 1)
            reasoning = re.sub(r"<<.*?>>", "", reasoning).strip()      # drop calculator annotations
            content = f"<think>\n{reasoning}\n</think>\n\nThe answer is {final_num.strip()}."
        else:
            content = re.sub(r"<<.*?>>", "", ans)
        return {"messages": [{"role": "user", "content": q}, {"role": "assistant", "content": content}]}

    def format_orca(ex):
        return {"messages": [{"role": "user", "content": clean_string(ex.get("question"))},
                             {"role": "assistant", "content": clean_string(ex.get("answer"))}]}

    def format_arc(ex):
        q = clean_string(ex.get("question"))
        ch = ex.get("choices", {})
        labels = ch.get("label", []) if isinstance(ch, dict) else []
        texts = ch.get("text", []) if isinstance(ch, dict) else []
        key = clean_string(ex.get("answerKey"))
        if not key:
            return {"messages": []}
        opts = "\n".join(f"{l}. {t}" for l, t in zip(labels, texts))
        return {"messages": [{"role": "user", "content": f"Question: {q}\n{opts}\nAnswer directly with the correct choice."},
                             {"role": "assistant", "content": f"The correct answer is {key}."}]}

    def format_ultrachat(ex):
        msgs = [{"role": m.get("role", "user"), "content": clean_string(m.get("content"))}
                for m in ex.get("messages", []) if m.get("content")]
        return {"messages": msgs}

    def prepare_sft_stream(ds, formatter, src):
        def fmt(ex):
            out = formatter(ex)
            out.setdefault("src", src)
            return out
        return (ds.map(fmt).select_columns(["messages", "src"])
                  .filter(lambda x: len(x.get("messages", [])) >= 2 and len(x["messages"][-1]["content"]) > 3))

    sources = [  # (name, stream, formatter, intended share: used only for the report below)
        ("med_o1", med_o1_stream, format_med_o1, 0.16),
        ("ii_med", ii_med_stream, format_ii_med, 0.15),
        ("medqa_train", medqa_stream, format_medqa, 0.11),
        ("medmcqa", medmcqa_stream, format_medmcqa, 0.14),
        ("pubmedqa", pubmedqa_stream, format_pubmedqa, 0.09),
        ("pubmed_causal", pubmed_causal_stream, format_pubmed_causal, 0.05),
        ("alpacare", alpacare_stream, format_alpacare, 0.06),
        ("gsm8k", gsm8k_stream, format_gsm8k, 0.08),
        ("orca_math", orca_math_stream, format_orca, 0.06),
        ("arc", arc_stream, format_arc, 0.04),
        ("ultrachat", ultrachat_stream, format_ultrachat, 0.06),
    ]
    assert abs(sum(s[3] for s in sources) - 1.0) < 1e-9

    # ============================================================ stage 1: one pass per source, all sources in parallel threads
    # (with MAX_COPIES=1 the final set is just the union of the sources, so no interleaving / recycling is needed)
    MAX_CHARS = 16_000     # a row this long cannot fit in TARGET_SEQ_LEN tokens (it would need >= 8 chars/token): skip before tokenizing

    def process_source(name, stream, formatter):
        out, local = [], collections.Counter()
        for rec in prepare_sft_stream(stream, formatter, name):
            messages, src = rec["messages"], rec["src"]
            if sum(len(m["content"]) for m in messages) > MAX_CHARS:
                local["skipped_length"] += 1
                continue
            try:
                text = data_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
            except Exception:  # noqa: BLE001
                local["skipped_template_error"] += 1
                continue
            n_tok = len(data_tokenizer.encode(text, add_special_tokens=False))
            if not (40 < n_tok <= TARGET_SEQ_LEN):
                local["skipped_length"] += 1
                continue
            key = hashlib.md5(json.dumps(messages, ensure_ascii=False).encode()).hexdigest()
            out.append((key, text, n_tok, src))
        return out, local

    results = {}
    with ThreadPoolExecutor(max_workers=len(sources)) as ex:
        futs = {ex.submit(process_source, n, s, f): n for n, s, f, _ in sources}
        for fut in as_completed(futs):
            name = futs[fut]
            results[name] = fut.result()
            print(f"  [{(time.perf_counter() - t_start) / 60:5.1f} min] finished {name:14s} {len(results[name][0]):>8,} candidate rows", flush=True)

    # ============================================================ stage 2: merge in fixed source order: dedupe -> contamination -> caps
    seen, kept_rows = set(), []
    st = collections.Counter()
    kept_by_src, tokens = collections.Counter(), 0
    for name, *_ in sources:
        out, local = results[name]
        st.update(local)
        for key, text, n_tok, src in tqdm(out, desc=f"merge {name}", leave=False):
            if key in seen:
                st["skipped_duplicate"] += 1
                continue
            seen.add(key)
            if CONTAM_FILTER and contaminated(text):
                st["skipped_contaminated"] += 1
                st[f"contaminated_{src}"] += 1
                continue
            if PUBMED_YES_NO_RATIO_CAP and src == "pubmedqa_yes" and \
                    kept_by_src["pubmedqa_yes"] > PUBMED_YES_NO_RATIO_CAP * max(kept_by_src["pubmedqa_no"], 50):
                st["skipped_pubmed_yes_cap"] += 1
                continue
            kept_rows.append({"text": text})
            kept_by_src[src] += 1
            tokens += n_tok

    ds_all = Dataset.from_list(kept_rows)
    split = ds_all.train_test_split(test_size=HOLDOUT, seed=SEED, shuffle=True)
    split.save_to_disk(CACHE_TRAIN_EVAL)
    stats = {"unique_rows": len(kept_rows), "tokens": tokens, "kept_by_source": dict(kept_by_src), "skips": dict(st),
             "max_copies": MAX_COPIES, "contam_filter": CONTAM_FILTER, "ngram": NG, "pubmed_cap": PUBMED_YES_NO_RATIO_CAP}
    json.dump(stats, open(os.path.join(CACHE_DIR, "stats.json"), "w"), indent=2)

    print(f"\nSFT C cache saved: {CACHE_TRAIN_EVAL}   (built in {(time.perf_counter() - t_start) / 60:.1f} min)")
    print(f"train {len(split['train']):,} | eval {len(split['test']):,} | {tokens / 1e6:.1f}M tokens")
    print("\nkept per source (compare with the intended shares):")
    for name, _, _, p in sources:
        n = sum(v for k, v in kept_by_src.items() if k == name or k.startswith(name + "_"))
        print(f"  {name:14s} intended {p * 100:4.0f}%   kept {n:>8,} ({100 * n / max(len(kept_rows), 1):4.1f}%)")
    print("\nPubMedQA labels kept:", {k: v for k, v in kept_by_src.items() if k.startswith("pubmedqa_")})
    print("skips:", dict(st))


In [ ]:
import torch
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name=BASE_MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.bfloat16,
    load_in_4bit=False,
    load_in_8bit=False,
    full_finetuning=False,
    offload_embedding=False,
)

if USE_DAPT:                      # matched DAPT arm only
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, DAPT_ADAPTER)
    model = model.merge_and_unload()
    print("DAPT adapter merged.")
else:
    print("No DAPT: training from the plain base.")

model.train()
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
text_tokenizer.padding_side = "right"
if text_tokenizer.pad_token_id is None:
    text_tokenizer.pad_token = text_tokenizer.eos_token
model.config.use_cache = False

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    modules_to_save=None,
    r=128,
    lora_alpha=256,
    lora_dropout=0.0,
    bias="none",
    use_dora=False,
    use_gradient_checkpointing=False,
    random_state=SEED,
)
model.print_trainable_parameters()          # SFT A showed 169,869,312 trainable parameters; this should match

# ---- guard: flash-linear-attention (fla) must be importable, or Qwen3.5's linear-attention layers use a very slow PyTorch fallback
import importlib.util, sys
if importlib.util.find_spec("fla") is None:
    print("[WARNING] flash-linear-attention is NOT installed -> training would sit at step 0. "
          "Run  pip install -U flash-linear-attention  and restart the session.")
else:
    _fast = {k: bool(v.is_fast_path_available) for k, v in sys.modules.items() if "qwen3_5" in k and hasattr(v, "is_fast_path_available")}
    print("fla installed | transformers fast-path flag:", _fast or "unknown")   # False can just mean causal-conv1d is missing: that is OK


In [ ]:
import gc, json, math, os, re, shutil, time
import torch
from datasets import load_from_disk
from transformers import TrainerCallback
from tqdm.auto import tqdm
from unsloth import UnslothTrainer, UnslothTrainingArguments
from unsloth.chat_templates import train_on_responses_only

gc.collect(); torch.cuda.empty_cache()

# ---- pre-flight: Qwen3.5's linear-attention layers need flash-linear-attention (fla). Without it transformers silently falls back to
#      a pure-PyTorch path whose backward pass is extremely slow (training sits at step 0). Install it, then RESTART the session.
import importlib.util, sys
assert importlib.util.find_spec("fla") is not None, \
    "flash-linear-attention is not installed: run  pip install -U flash-linear-attention  then Runtime -> Restart session"
_fast = {k: bool(v.is_fast_path_available) for k, v in sys.modules.items() if "qwen3_5" in k and hasattr(v, "is_fast_path_available")}
print("fla installed | transformers fast-path flag:", _fast or "unknown")   # the flag can read False when only causal-conv1d is missing: that is OK

BASELINE_EVAL = False       # the pre-SFT baseline on this exact split/model was already measured: 1.3084 (PPL 3.70); True re-measures it (~70 s)

DRIVE_CHECKPOINTS_DIR = os.path.join(DRIVE_ROOT, "checkpoints")
DRIVE_TENSORBOARD_DIR = os.path.join(DRIVE_ROOT, "tensorboard_logs")
DRIVE_FINAL_DIR = os.path.join(DRIVE_ROOT, "final_adapter")
for d in (DRIVE_CHECKPOINTS_DIR, DRIVE_TENSORBOARD_DIR, DRIVE_FINAL_DIR):
    os.makedirs(d, exist_ok=True)
os.environ["TENSORBOARD_LOGGING_DIR"] = DRIVE_TENSORBOARD_DIR

sft_data = load_from_disk(os.path.join(CACHE_DIR, "split"))
train_dataset, eval_dataset = sft_data["train"], sft_data["test"]
print(f"Train {len(train_dataset):,} | Eval {len(eval_dataset):,}")

eff_batch = BATCH_SIZE * GRAD_ACCUM
steps = min(TOTAL_STEPS, len(train_dataset) // eff_batch)
if steps < TOTAL_STEPS:
    print(f"[note] dataset only supports {steps} steps of {eff_batch}; max_steps reduced from {TOTAL_STEPS} to {steps}")


class PerplexityLoggingCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics and "eval_loss" in metrics:
            ppl = math.exp(metrics["eval_loss"]) if metrics["eval_loss"] < 50 else float("inf")
            metrics["eval_perplexity"] = ppl
            print(f"\n>>> [Step {state.global_step}] Eval CE Loss: {metrics['eval_loss']:.4f} | Eval Perplexity: {ppl:.2f}", flush=True)


class SFTProgressBar(TrainerCallback):
    """One live tqdm bar (steps, elapsed, ETA, s/step, last loss, peak memory) plus a history line every 50 steps."""
    def __init__(self):
        self.bar, self.t0, self.start_step, self.last_loss = None, None, 0, "NA"

    def on_train_begin(self, args, state, control, **kwargs):
        self.t0 = time.perf_counter()
        self.start_step = state.global_step                 # non-zero when resuming
        self.bar = tqdm(total=state.max_steps, initial=state.global_step, desc="SFT", unit="step",
                        dynamic_ncols=True, smoothing=0.05, leave=True)
        tqdm.write(f"[train] started at step {self.start_step}")

    def on_step_end(self, args, state, control, **kwargs):
        if self.bar is None:
            return
        if state.global_step > self.bar.n:
            self.bar.update(state.global_step - self.bar.n)
        self.bar.set_postfix(loss=self.last_loss, peak=f"{torch.cuda.max_memory_allocated() / 2**30:.1f}G", refresh=True)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if self.bar is None:                                # on_log can fire during evaluate(), before training starts
            return
        if logs and "loss" in logs:
            self.last_loss = f"{logs['loss']:.3f}"
        done = state.global_step - self.start_step
        if done and state.global_step % 50 == 0:
            el = time.perf_counter() - self.t0
            sps = el / done
            eta = sps * (state.max_steps - state.global_step)
            tqdm.write(f"Step {state.global_step}/{state.max_steps} | {el / 60:.0f} min | ETA {eta / 3600:.2f} h | "
                       f"{sps:.2f} s/step | loss {self.last_loss}")

    def on_train_end(self, args, state, control, **kwargs):
        if self.bar is not None:
            self.bar.close()


training_args = UnslothTrainingArguments(
    output_dir=DRIVE_CHECKPOINTS_DIR,
    run_name=RUN_NAME,
    report_to="tensorboard",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    per_device_eval_batch_size=BATCH_SIZE,   # 16 was slower (1:56 vs 1:10): bigger batches pad more
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    # to test length grouping use train_sampling_strategy="group_by_length" (this transformers version ignores group_by_length)
    disable_tqdm=True,                       # our bar below replaces the notebook widget that did not render
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_steps=steps,
    optim="adamw_torch_fused",
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_grad_norm=1.0,
    weight_decay=0.01,
    bf16=True,
    fp16=False,
    gradient_checkpointing=GRAD_CKPT,
    dataloader_num_workers=0,                # 2 workers can hang when forked after the threaded data build in the same process
    dataloader_pin_memory=True,
    save_strategy="steps",
    save_steps=EVAL_STEPS,
    save_total_limit=2,
    seed=SEED,
)

trainer = UnslothTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    callbacks=[PerplexityLoggingCallback(), SFTProgressBar()],
)
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")

# ---- auto-resume from the newest checkpoint on Drive (needs the same cache, the same settings, and cells 1, 3, 4 re-run)
ckpts = [d for d in os.listdir(DRIVE_CHECKPOINTS_DIR) if re.fullmatch(r"checkpoint-\d+", d)]
resume_path = os.path.join(DRIVE_CHECKPOINTS_DIR, max(ckpts, key=lambda d: int(d.split("-")[1]))) if ckpts else None
print("Resuming from:", resume_path or "scratch")

if resume_path is None and BASELINE_EVAL:    # baseline eval only on a fresh run
    print("Computing the pre-SFT baseline eval loss...")
    base_loss = trainer.evaluate()["eval_loss"]
    print(f"Pre-SFT baseline CE loss: {base_loss:.4f} | PPL {math.exp(base_loss):.2f}")
    # SFT A on the DAPT-merged model started at 1.2597 (its own held-out split, so not exactly comparable).

trainer.train(resume_from_checkpoint=resume_path)

print(f"Training complete. Saving the adapter to {DRIVE_FINAL_DIR} ...")
model.save_pretrained(DRIVE_FINAL_DIR)
text_tokenizer.save_pretrained(DRIVE_FINAL_DIR)

# provenance: keep the exact data stats and run settings next to the weights
shutil.copy(os.path.join(CACHE_DIR, "stats.json"), os.path.join(DRIVE_ROOT, "data_stats.json"))
json.dump(dict(run=RUN_NAME, base=BASE_MODEL_ID, use_dapt=USE_DAPT, steps=steps, eff_batch=eff_batch, lr=LEARNING_RATE,
               seed=SEED, grad_ckpt=GRAD_CKPT, batch_size=BATCH_SIZE, grad_accum=GRAD_ACCUM, lora_r=128, lora_alpha=256),
          open(os.path.join(DRIVE_ROOT, "run_config.json"), "w"), indent=2)
print("Saved.")


In [ ]:
import os, shutil
from safetensors.torch import load_file, save_file

SRC = os.path.join(DRIVE_ROOT, "final_adapter")
DST = os.path.join(DRIVE_ROOT, "final_adapter_peft_fixed")
os.makedirs(DST, exist_ok=True)
for f in os.listdir(SRC):
    p = os.path.join(SRC, f)
    if os.path.isfile(p) and not f.endswith((".safetensors", ".bin", ".pt")):
        shutil.copy2(p, DST)

# Unsloth saves  base_model.model.model.language_model.layers.N...  (multimodal wrapper names);
# the eval / RLVR loaders use AutoModelForCausalLM  ->  base_model.model.model.layers.N...  (same as the working SFT A adapter)
tensors = load_file(os.path.join(SRC, "adapter_model.safetensors"))
def fix(k):
    k = k.replace("base_model.model.model.model.", "base_model.model.model.", 1)   # doubled prefix, if present
    return k.replace(".language_model.", ".", 1)
fixed = {fix(k): v for k, v in tensors.items()}
assert len(fixed) == len(tensors), "key collision while remapping"
print(f"remapped {sum(fix(k) != k for k in tensors)} of {len(tensors)} keys")
print("before:", sorted(tensors)[0])
print("after :", sorted(fixed)[0])
save_file(fixed, os.path.join(DST, "adapter_model.safetensors"), metadata={"format": "pt"})
print("Saved:", DST)

# ---- verify: load it the way the eval does (CPU, AutoModelForCausalLM + PeftModel) and make sure EVERY lora_B is non-zero.
#      A zero lora_B means the keys did not match and the adapter would be a silent no-op.
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM
chk = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, torch_dtype=torch.bfloat16, device_map="cpu", trust_remote_code=True)
chk = PeftModel.from_pretrained(chk, DST, is_trainable=False)
b = {n: p.float().abs().sum().item() for n, p in chk.named_parameters() if "lora_B" in n}
zero = [n for n, v in b.items() if v == 0]
print(f"lora_B tensors: {len(b)} | all-zero: {len(zero)}")
assert len(b) == 128 and not zero, f"adapter did not load correctly, e.g. {zero[:2]}"
print("OK: adapter keys match the model; the weights loaded.")
del chk; import gc; gc.collect()
